In [0]:
%sql
create catalog if not exists retail;


In [0]:
%sql

create catalog if not exists fmcg;
use catalog fmcg;

create schema if not exists gold;
create schema if not exists silver;
create schema if not exists bronze;
use schema bronze;

In [0]:
%python

# Import Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

# Define variables
start_date = '2024-01-01'
end_date = '2025-12-31'

df_dates= (spark.sql
            (f"""select
            explode(
                sequence(
                    to_date('{start_date}'), 
                    to_date('{end_date}'), 
                    interval 1 month
                    )
                    ) as month_start_date
                """)
            )

# Add necessary analytics columns
df_dates = (df_dates
    .withColumn("date_key", date_format("month_start_date","yyyyMM").cast("int"))
    .withColumn("year", year("month_start_date"))
    .withColumn("month", month("month_start_date"))
    .withColumn("month_name", date_format("month_start_date","MMMM"))
    .withColumn("month_short_name", date_format("month_start_date","MMM"))
    .withColumn("quarter", concat(lit("Q"),quarter("month_start_date")).cast("string"))
    .withColumn("year_quarter", concat(col("year"), lit("-Q"), quarter("month_start_date")).cast("string"))
    )

display(df_dates)

# save as table
df_dates.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("fmcg.gold.dim_dates")

month_start_date,date_key,year,month,month_name,month_short_name,quarter,year_quarter
2024-01-01,202401,2024,1,January,Jan,Q1,2024-Q1
2024-02-01,202402,2024,2,February,Feb,Q1,2024-Q1
2024-03-01,202403,2024,3,March,Mar,Q1,2024-Q1
2024-04-01,202404,2024,4,April,Apr,Q2,2024-Q2
2024-05-01,202405,2024,5,May,May,Q2,2024-Q2
2024-06-01,202406,2024,6,June,Jun,Q2,2024-Q2
2024-07-01,202407,2024,7,July,Jul,Q3,2024-Q3
2024-08-01,202408,2024,8,August,Aug,Q3,2024-Q3
2024-09-01,202409,2024,9,September,Sep,Q3,2024-Q3
2024-10-01,202410,2024,10,October,Oct,Q4,2024-Q4
